# Logging

<a target="_blank" href="https://colab.research.google.com/github/avr2002/cloud-engineering-project/blob/feat/logging/notebooks/logging.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

- Logging is a means of tracking "events" when your application runs. 

- An *event* can be anything of interest that happens during the execution of your program like occurance of an error, a simple infomatic message like your program started or you API call was successful etc.

  - Events are logged with a descriptive message which optionally can have associated application data.

  - Events also have an importance which you as the developer ascribes it; the importance can also be called the **level or severity**.

| Level     | When it’s used                                                                                  |
|-----------|-------------------------------------------------------------------------------------------------|
| `DEBUG`   | Detailed information, typically of interest only when diagnosing problems.                      |
| `INFO`    | Confirmation that things are working as expected.                                               |
| `WARNING` | An indication that something unexpected happened, or indicative of some problem in the near future (e.g. ‘disk space low’). The software is still working as expected. |
| `ERROR`   | Due to a more serious problem, the software has not been able to perform some function.         |
| `CRITICAL`| A serious error, indicating that the program itself may be unable to continue running.          |

<p align="right"><i>Source: <a href="https://docs.python.org/3/howto/logging.html">Python Logging Docs</a></i></p>



Setting up logging can be hard, but in this notebook we will go through different tools and libraries that can help you set up logging in your application.

In [45]:
# Install the logging libraries

%pip install -q loguru aws-lambda-powertools mangum nest_asyncio

Note: you may need to restart the kernel to use updated packages.


## Python's `logging` module

In [7]:
# Setup logging in the simplest way possible

import os
import sys
import logging


# get log level from environment variable
LOG_LEVEL = os.environ.get("LOG_LEVEL", "INFO").upper()

# configure logger object
logging.basicConfig(
    format="{asctime} | {levelname} | {name}:{lineno}:{funcName} | {message}",
    style="{",  # uses {} as placeholders
    level=LOG_LEVEL,
    stream=sys.stdout,  # where to write the log messages, in this case stdout or console
)
logger = logging.getLogger(__name__)  # create logger object with the name of the current module

# log some messages
logger.debug("This is a debug message")  # this will not be printed because the log level is set to INFO
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")
# logger.exception("This is an exception message")

2024-08-09 23:35:48,389 | INFO | __main__:22:<module> | This is an info message
2024-08-09 23:35:48,390 | WARNING | __main__:23:<module> | This is a warning message
2024-08-09 23:35:48,392 | ERROR | __main__:24:<module> | This is an error message
2024-08-09 23:35:48,392 | CRITICAL | __main__:25:<module> | This is a critical message


The logging library takes a modular approach and offers several categories of components: *loggers, handlers, filters, and formatters*.

- ***Loggers*** expose the interface that application code directly uses.
- ***Handlers*** send the log records (created by loggers) to the appropriate destination.
- ***Filters*** provide a finer grained facility for determining which log records to output.
- ***Formatters*** specify the layout of log records in the final output.

<p align="right"><i>Source: <a href="https://docs.python.org/3/howto/logging.html#advanced-logging-tutorial">Advanced Python Logging Docs</a></i></p>

In [1]:
# Advanced logging setup with a console handler and a formatter

import logging
import os
import sys


LOG_LEVEL = os.environ.get("LOG_LEVEL", "INFO").upper()

# create the logger
logger = logging.getLogger(__name__)
logger.setLevel(LOG_LEVEL)

# create a console handler and set its log level
ch = logging.StreamHandler(stream=sys.stderr)
ch.setLevel(LOG_LEVEL)

# create a formatter
formatter = logging.Formatter("{asctime} | {levelname} | {name}:{lineno}:{funcName} | {message}", style="{")
ch.setFormatter(formatter)  # add the formatter to the console handler

# add the console handler to the logger
logger.addHandler(ch)


# log some messages
logger.debug("This is a debug message")
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")

2024-08-09 23:39:19,011 | INFO | __main__:28:<module> | This is an info message
2024-08-09 23:39:19,013 | WARNING | __main__:29:<module> | This is a warning message
2024-08-09 23:39:19,014 | ERROR | __main__:30:<module> | This is an error message
2024-08-09 23:39:19,014 | CRITICAL | __main__:31:<module> | This is a critical message


In [3]:
import random
import time


def process_user_actions(user_id, actions):
    """
    Simulates processing a list of user actions.

    Parameters:
    user_id (int): The ID of the user.
    actions (list of str): A list of actions to process.

    Returns:
    None
    """

    # Simulate different levels of logging for demonstration
    logger.info(f"Start processing actions for user {user_id}.")

    for index, action in enumerate(actions):
        try:
            logger.debug(f"Processing action {index + 1}/{len(actions)}: {action}.")

            # Simulate processing time
            processing_time = random.uniform(0.1, 0.5)
            time.sleep(processing_time)

            # Simulate a warning scenario
            if "warning" in action:
                logger.warning(f"Potential issue detected in action: {action}.")

            # Simulate an error scenario
            if "error" in action:
                raise ValueError(f"Failed to process action: {action}.")

            logger.info(f"Successfully processed action: {action}.")

        except Exception as e:
            logger.error(f"Error processing action {action}: {str(e)}")
            logger.exception(f"Exception details: {str(e)}")
            logger.debug("Continuing with the next action.")

    logger.info(f"Finished processing actions for user {user_id}.")

In [ ]:
# Sample usage
if __name__ == "__main__":
    user_id = 123
    actions = ["login", "view_page", "add_to_cart", "warning: slow network", "checkout", "error: payment failed"]

    process_user_actions(user_id, actions)

## [`loguru`](https://loguru.readthedocs.io/en/stable/overview.html)

>The main concept of Loguru is that there is one and only one logger. No Handler, no Formatter, no Filter: one function to rule them all.

In [44]:
from loguru import logger

# log some messages
logger.debug("This is a debug message")
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")
# logger.exception("This is an exception message")

2024-08-10 20:04:36.307 | DEBUG    | __main__:<module>:4 - This is a debug message
2024-08-10 20:04:36.308 | INFO     | __main__:<module>:5 - This is an info message
2024-08-10 20:04:36.309 | WARNING  | __main__:<module>:6 - This is a warning message
2024-08-10 20:04:36.310 | ERROR    | __main__:<module>:7 - This is an error message
2024-08-10 20:04:36.311 | CRITICAL | __main__:<module>:8 - This is a critical message


In [18]:
import sys
import json
from loguru import logger


def serialize_extra_keys(record):
    extra = record["extra"]
    if extra:
        record["extra"] = json.dumps(extra)
    return record


logging_config = {
    "handlers": [
        {
            "sink": sys.stdout,
            "format": "<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{name}:{function}:{line}</cyan> | <level>{message}</level> | <level>{extra}</level>",
            "level": "INFO",
            "filter": serialize_extra_keys,
            "colorize": True,
        },
    ],
}

# Remove the default logger config and add custom configurations
logger.remove()
logger.configure(**logging_config)


# log some messages
logger.debug("This is a debug message")
logger.success("This is a success message")
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")
logger.info(
    "Adding extra data to log messages",
    extra={"extra_key": "extra_value", "another_key": "another_value", "user_id": 12345},
)
# logger.exception("This is an exception message", extra={"user_id": 12345})

2024-08-09 23:59:44 | SUCCESS  | __main__:<module>:31 | This is a success message | {}
2024-08-09 23:59:44 | INFO     | __main__:<module>:32 | This is an info message | {}
2024-08-09 23:59:44 | WARNING  | __main__:<module>:33 | This is a warning message | {}
2024-08-09 23:59:44 | ERROR    | __main__:<module>:34 | This is an error message | {}
2024-08-09 23:59:44 | CRITICAL | __main__:<module>:35 | This is a critical message | {}
2024-08-09 23:59:44 | INFO     | __main__:<module>:36 | Adding extra data to log messages | {"extra": {"extra_key": "extra_value", "another_key": "another_value", "user_id": 12345}}


In [20]:
# Sample usage
if __name__ == "__main__":
    user_id = 123
    actions = ["login", "view_page", "add_to_cart", "warning: slow network", "checkout", "error: payment failed"]

    process_user_actions(user_id, actions)

2024-08-10 00:01:24 | INFO     | __main__:process_user_actions:17 | Start processing actions for user 123. | {}
2024-08-10 00:01:25 | INFO     | __main__:process_user_actions:35 | Successfully processed action: login. | {}
2024-08-10 00:01:25 | INFO     | __main__:process_user_actions:35 | Successfully processed action: view_page. | {}
2024-08-10 00:01:25 | INFO     | __main__:process_user_actions:35 | Successfully processed action: add_to_cart. | {}
2024-08-10 00:01:26 | WARNING  | __main__:process_user_actions:29 | Potential issue detected in action: warning: slow network. | {}
2024-08-10 00:01:26 | INFO     | __main__:process_user_actions:35 | Successfully processed action: warning: slow network. | {}
2024-08-10 00:01:26 | INFO     | __main__:process_user_actions:35 | Successfully processed action: checkout. | {}
2024-08-10 00:01:26 | ERROR    | __main__:process_user_actions:38 | Error processing action error: payment failed: Failed to process action: error: payment failed. | {}
202

## [`aws-lambda-powertools`](https://docs.powertools.aws.dev/lambda/python/latest/core/logger/)

>Logging utilities for AWS Lambda functions.

In [40]:
import os
from typing import Any
from aws_lambda_powertools import Logger
from aws_lambda_powertools.utilities.typing import LambdaContext
import os

LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO")

logger = Logger(level=LOG_LEVEL)


@logger.inject_lambda_context(clear_state=True) # clear_state=True will clear the logger state for each invocation
def handler(event: dict, context: LambdaContext) -> Any:
    try:
        logger.info("This is an info message")
        logger.debug("This is a debug message")
        logger.warning("This is a warning message")
        
        # Add extra data to future log messages
        logger.append_keys(extra_data={"user_id": 12345})
        
        logger.error("This is an error message")
        logger.critical("This is a critical message")

        # raise ValueError("This is a sample exception")
        
        logger.info("Finished processing the event.")
        return {"statusCode": 200, "body": "Hello, World!"}
    except Exception as e:
        logger.exception("An error occurred during execution")
        return {"statusCode": 500, "body": "Internal Server Error"}


# Simulate the Lambda context (optional)
class Context:
    def __init__(self):
        self.function_name = "test-function"
        self.memory_limit_in_mb = 128
        self.invoked_function_arn = "arn:aws:lambda:us-west-2:123456789012:function:test-function"
        self.aws_request_id = "fake-request-id"


if __name__ == "__main__":
    context = Context()

    # Call the handler function as Lambda would
    response = handler({}, context)

{"level":"INFO","location":"handler:15","message":"This is an info message","timestamp":"2024-08-10 12:40:40,372+0530","service":"service_undefined","cold_start":false,"function_name":"test-function","function_memory_size":128,"function_arn":"arn:aws:lambda:us-west-2:123456789012:function:test-function","function_request_id":"fake-request-id"}
{"level":"WARNING","location":"handler:17","message":"This is a warning message","timestamp":"2024-08-10 12:40:40,373+0530","service":"service_undefined","cold_start":false,"function_name":"test-function","function_memory_size":128,"function_arn":"arn:aws:lambda:us-west-2:123456789012:function:test-function","function_request_id":"fake-request-id"}
{"level":"ERROR","location":"handler:22","message":"This is an error message","timestamp":"2024-08-10 12:40:40,374+0530","service":"service_undefined","cold_start":false,"function_name":"test-function","function_memory_size":128,"function_arn":"arn:aws:lambda:us-west-2:123456789012:function:test-functi

In [35]:
import os
import sys
from aws_lambda_powertools import Logger
from aws_lambda_powertools.utilities.typing import LambdaContext


LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO")


# Initialize the logger
logger = Logger(stream=sys.stdout, level=LOG_LEVEL)


@logger.inject_lambda_context
def handler(event: dict, context: LambdaContext):
    logger.info("Starting to process the event.")

    user_id = event.get("user_id", "unknown")
    actions = event.get("actions", [])

    logger.append_keys(user_id=user_id)  # Adding context to all logs
    process_user_actions(user_id, actions)
    return {"statusCode": 200, "body": "Handler executed successfully."}


# Simulate an event
if __name__ == "__main__":
    sample_event = {
        "user_id": 123,
        "actions": ["login", "view_page", "warning: slow network", "error: payment failed"],
    }
    context = Context()

    # Call the handler function as Lambda would
    response = handler(sample_event, context)

{"level":"INFO","location":"handler:16","message":"Starting to process the event.","timestamp":"2024-08-10 12:37:13,398+0530","service":"service_undefined","cold_start":false,"function_name":"test-function","function_memory_size":128,"function_arn":"arn:aws:lambda:us-west-2:123456789012:function:test-function","function_request_id":"fake-request-id","extra_data":{"user_id":12345}}
{"level":"INFO","location":"process_user_actions:17","message":"Start processing actions for user 123.","timestamp":"2024-08-10 12:37:13,399+0530","service":"service_undefined","cold_start":false,"function_name":"test-function","function_memory_size":128,"function_arn":"arn:aws:lambda:us-west-2:123456789012:function:test-function","function_request_id":"fake-request-id","extra_data":{"user_id":12345},"user_id":123}
{"level":"INFO","location":"process_user_actions:35","message":"Successfully processed action: login.","timestamp":"2024-08-10 12:37:13,535+0530","service":"service_undefined","cold_start":false,"f

## Mocked FastAPI Application

In [1]:
import sys
from fastapi import FastAPI, Request
from loguru import logger as loguru_logger
from aws_lambda_powertools import Logger as PowertoolsLogger

APP = FastAPI()

# Set up Loguru
loguru_logger.remove()
loguru_logger.add(sys.stdout, format="{time:YYYY-MM-DD HH:mm:ss,SSS} | {level} | {message}", level="DEBUG")

# Set up AWS Lambda Powertools Logger
powertools_logger = PowertoolsLogger()

@APP.post("/echo")
async def echo(request: Request):
    data = await request.json()
    loguru_logger.info("Loguru: Received request with data: {}", data)
    powertools_logger.info("Powertools: Received request with data", extra={"data": data})
    return {"echo": data}

In [8]:
import json

# adapter for FastAPI and lambda handler
from mangum import Mangum

# mock s3
from moto import mock_aws

# make mangum's asyncio work in jupyter
import nest_asyncio


S3_BUCKET_NAME = "some-bucket"  # can be fake since we're mocking S3
AWS_REGION = "us-east-1"

In [9]:
from contextlib import contextmanager
import os
from mangum.types import LambdaContext

# custom context class for Lambda
class MockedLambdaContext(LambdaContext):
    function_name: str = "test_function"
    function_version: str = "1"
    invoked_function_arn: str = f"arn:aws:lambda:{AWS_REGION}:123456789012:function:test_function"
    memory_limit_in_mb: int = 128
    aws_request_id: str = "unique-request-id"
    log_group_name: str = "/aws/lambda/test_function"
    log_stream_name: str = "2021/03/26/[$LATEST]abcdef1234567890abcdef"
    identity: str = None
    client_context: str = None

    def get_remaining_time_in_millis(self) -> int:
        return 30000  # 30 seconds

@contextmanager
def mock_aws_and_env_vars():
    with mock_aws():
        os.environ["AWS_REGION"] = "mock-region"
        os.environ["AWS_ACCESS_KEY_ID"] = "mock-access-key-id"
        os.environ["AWS_SECRET_ACCESS_KEY"] = "mock-secret-access-key"
        os.environ.pop("AWS_SESSION_TOKEN", None)
        os.environ.pop("AWS_PROFILE", None)
        yield
        
def simulate_lambda_invocation(event: dict, context: LambdaContext):
    """Function to simulate AWS Lambda invocation."""
    handler = Mangum(APP, lifespan="off")
    return handler(event, context)

In [10]:
# Enable nested event loops for Jupyter
nest_asyncio.apply()

# Simulate a Lambda invocation
event = {
    "httpMethod": "POST",
    "path": "/echo",
    "body": json.dumps({"message": "Hello, World!"}),
    "headers": {
        "Content-Type": "application/json",
    },
    "isBase64Encoded": False,
    "requestContext": {
        "httpMethod": "POST",
        "path": "/echo",
    },
    "resource": "/echo",
    "queryStringParameters": None,
    "pathParameters": None,
    "stageVariables": None
}

with mock_aws_and_env_vars():
    context = MockedLambdaContext()
    response = simulate_lambda_invocation(event, context)
    print(response)


2024-08-10 21:19:42,436 | INFO | Loguru: Received request with data: {'message': 'Hello, World!'}
{"level":"INFO","location":"echo:19","message":"Powertools: Received request with data","timestamp":"2024-08-10 21:19:42,437+0530","service":"service_undefined","data":{"message":"Hello, World!"}}
{'statusCode': 200, 'headers': {'content-length': '36', 'content-type': 'application/json'}, 'multiValueHeaders': {}, 'body': '{"echo":{"message":"Hello, World!"}}', 'isBase64Encoded': False}


In [11]:
# from pyngrok import ngrok

# ngrok_tunnel = ngrok.connect(8000)
# ngrok_tunnel


# import nest_asyncio
# import uvicorn

# nest_asyncio.apply()
# uvicorn.run(app, port=8000)

In [12]:
# Mocking API Gateway with Moto


from fastapi.testclient import TestClient
import boto3
from moto import mock_aws
from mangum import Mangum


# Wrap the FastAPI app with Mangum to make it AWS Lambda compatible
handler = Mangum(APP, lifespan="off")

# Using the existing FastAPI app and handler from previous code
client = TestClient(APP)

@mock_aws
def test_lambda_invocation():
    # Setup the API Gateway client with moto
    apigateway = boto3.client("apigateway", region_name=AWS_REGION)
    
    # Mock API creation
    api = apigateway.create_rest_api(name="test_api")
    resource_id = apigateway.get_resources(restApiId=api['id'])['items'][0]['id']
    
    # Mock POST method
    apigateway.put_method(
        restApiId=api['id'],
        resourceId=resource_id,
        httpMethod="POST",
        authorizationType="NONE"
    )
    # Mock Lambda function
    apigateway.put_integration(
        restApiId=api['id'],
        resourceId=resource_id,
        httpMethod="POST",
        type="AWS_PROXY",
        integrationHttpMethod="POST",
        uri=f"arn:aws:apigateway:us-west-2:lambda:path/2015-03-31/functions/{handler}/invocations"
    )
    
    # Simulate sending a POST request to the API Gateway
    response = client.post("/echo", json={"message": "Hello, world!"})
    
    # print(response.json())
    
    # Assertions to check if everything worked correctly
    assert response.status_code == 200
    assert response.json() == {"echo": {"message": "Hello, world!"}}

# Run the test
test_lambda_invocation()

2024-08-10 21:19:44,007 | INFO | Loguru: Received request with data: {'message': 'Hello, world!'}
{"level":"INFO","location":"echo:19","message":"Powertools: Received request with data","timestamp":"2024-08-10 21:19:44,008+0530","service":"service_undefined","data":{"message":"Hello, world!"}}
